# Predicting Numbness Using Health Data

## Overview
This project uses health and nutritional data to predict whether a person experiences numbness and tingling using machine learning.

## Data
The dataset includes vitamin levels, BMI, age, and health symptoms for 4000 individuals.

## Analysis
Lower vitamin B12 levels were strongly associated with numbness based on exploratory data analysis.

In [27]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
%matplotlib inline


df = pd.read_csv("vitamin_deficiency_disease_dataset_20260123.csv")

In [28]:
df.head()

,age,gender,bmi,smoking_status,alcohol_consumption,exercise_level,diet_type,sun_exposure,income_level,latitude_region,...,has_night_blindness,has_fatigue,has_bleeding_gums,has_bone_pain,has_muscle_weakness,has_numbness_tingling,has_memory_problems,has_pale_skin,disease_diagnosis,has_multiple_deficiencies
0,79,Male,24.8,Former,NaN,Active,Vegetarian,High,High,Mid,...,0,0,0,0,0,0,0,0,Healthy,0
1,77,Female,39.9,Former,Moderate,Light,Omnivore,Low,Low,Low,...,0,0,0,1,0,0,0,0,Rickets_Osteomalacia,0
2,24,Male,26.4,Former,Heavy,Moderate,Omnivore,Low,High,High,...,1,0,0,0,0,0,0,0,Healthy,0
3,69,Male,23.1,Never,Heavy,Moderate,Vegetarian,High,Low,Low,...,0,0,0,0,0,1,1,0,Anemia,0
4,63,Male,29.6,Never,NaN,Moderate,Vegetarian,Moderate,High,Low,...,0,0,0,0,0,0,0,0,Healthy,0


In [29]:
df = df.replace({0: False, 1: True})

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        4000 non-null   int64  
 1   gender                     4000 non-null   str    
 2   bmi                        4000 non-null   float64
 3   smoking_status             4000 non-null   str    
 4   alcohol_consumption        2722 non-null   str    
 5   exercise_level             4000 non-null   str    
 6   diet_type                  4000 non-null   str    
 7   sun_exposure               4000 non-null   str    
 8   income_level               4000 non-null   str    
 9   latitude_region            4000 non-null   str    
 10  vitamin_a_percent_rda      4000 non-null   float64
 11  vitamin_c_percent_rda      4000 non-null   float64
 12  vitamin_d_percent_rda      4000 non-null   float64
 13  vitamin_e_percent_rda      4000 non-null   float64
 14  vit

In [31]:
df.describe()

,age,bmi,vitamin_a_percent_rda,vitamin_c_percent_rda,vitamin_d_percent_rda,vitamin_e_percent_rda,vitamin_b12_percent_rda,folate_percent_rda,calcium_percent_rda,iron_percent_rda,hemoglobin_g_dl,serum_vitamin_d_ng_ml,serum_vitamin_b12_pg_ml,serum_folate_ng_ml
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,50.766250,26.105325,90.770850,89.199075,72.216388,89.946125,62.743225,90.382275,82.586300,76.211125,14.027200,21.744050,255.514725,10.831725
std,19.306237,4.922746,37.083534,37.505809,42.831000,37.444649,37.396636,37.563684,36.359131,33.226984,1.665629,13.768642,158.934806,4.965111
min,18.000000,15.000000,10.000000,10.000000,7.000000,10.000000,10.000000,10.000000,10.000000,10.000000,8.200000,5.000000,100.000000,2.000000
25%,34.000000,22.800000,62.600000,60.975000,40.730000,61.700000,32.700000,61.800000,55.600000,51.400000,12.900000,11.700000,121.200000,7.100000
50%,51.000000,26.200000,85.500000,83.500000,62.270000,84.050000,55.600000,84.800000,77.100000,71.250000,14.100000,18.400000,214.850000,10.000000
75%,67.000000,29.400000,115.300000,113.000000,93.317500,114.100000,84.500000,115.200000,105.300000,95.900000,15.100000,27.900000,338.400000,13.800000
max,84.000000,45.000000,219.000000,250.000000,275.600000,237.600000,243.600000,226.600000,232.700000,211.400000,18.000000,80.000000,1138.100000,25.000000


In [32]:
df[["vitamin_b12_percent_rda","has_numbness_tingling"]].head()

,vitamin_b12_percent_rda,has_numbness_tingling
0,102.5,False
1,62.6,False
2,136.2,False
3,31.8,True
4,72.6,False


In [33]:
df.groupby("has_numbness_tingling")["vitamin_b12_percent_rda"].mean()

has_numbness_tingling
False    80.206435
True     30.489110
Name: vitamin_b12_percent_rda, dtype: float64

In [34]:
df["has_numbness_tingling"].value_counts()

has_numbness_tingling
False    2595
True     1405
Name: count, dtype: int64

In [35]:
df["b12_risk"] = df["vitamin_b12_percent_rda"] < 50
df[["vitamin_b12_percent_rda", "b12_risk", "has_numbness_tingling"]].head(10)

,vitamin_b12_percent_rda,b12_risk,has_numbness_tingling
0,102.5,False,False
1,62.6,False,False
2,136.2,False,False
3,31.8,True,True
4,72.6,False,False
5,21.2,True,True
6,27.0,True,False
7,50.0,False,False
8,122.8,False,False
9,125.9,False,False


In [36]:
df.groupby("b12_risk")["has_numbness_tingling"].mean()

b12_risk
False         0.0
True     0.798295
Name: has_numbness_tingling, dtype: object

In [37]:
df.groupby("has_fatigue")["iron_percent_rda"].mean()

has_fatigue
False    86.691276
True     47.767781
Name: iron_percent_rda, dtype: float64

In [38]:
df["high_risk"] = (df["vitamin_b12_percent_rda"] < 50) & (df["iron_percent_rda"] < 50)
df[["vitamin_b12_percent_rda", "iron_percent_rda", "high_risk"]].head(10)

,vitamin_b12_percent_rda,iron_percent_rda,high_risk
0,102.5,97.4,False
1,62.6,102.5,False
2,136.2,86.4,False
3,31.8,60.8,False
4,72.6,71.9,False
5,21.2,79.0,False
6,27.0,54.7,False
7,50.0,120.4,False
8,122.8,84.9,False
9,125.9,40.3,False


In [39]:
df["high_risk"].value_counts()

high_risk
False    3364
True      636
Name: count, dtype: int64

In [40]:
df.groupby("high_risk")["symptoms_count"].mean()

high_risk
False     1.38912
True     4.396226
Name: symptoms_count, dtype: object

## Model
A logistic regression model was trained using vitamin B12, iron, BMI, and age to predict numbness.

In [41]:
from sklearn.model_selection import train_test_split

X = df[["vitamin_b12_percent_rda", "iron_percent_rda", "bmi", "age"]]
y = df["has_numbness_tingling"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [42]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((3200, 4), (800, 4), (3200,), (800,))

In [43]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

ValueError: Unknown label type: unknown. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
rf_predictions = rf_model.predict(X_test)

# Evaluation
print("Random Forest Accuracy:", accuracy_score(y_test, rf_predictions))
print(classification_report(y_test, rf_predictions))

In [ ]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_predictions))

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(by="importance", ascending=False)
print(feature_importance)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["feature"], feature_importance["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
logistic_coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

logistic_coefficients = logistic_coefficients.sort_values(by="coefficient", ascending=False)
print(logistic_coefficients)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(logistic_coefficients["feature"], logistic_coefficients["coefficient"])
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Logistic Regression Coefficients")
plt.gca().invert_yaxis()
plt.show()

## Model Insights

The Random Forest model helped identify which health factors contributed most to predicting numbness. This made the project more than just a prediction task because it also provided insight into the drivers behind the outcome. Logistic Regression gave an interpretable baseline, while Random Forest offered a comparison model with feature importance analysis.

In [ ]:
def predict_numbness(input_dict):
    input_df = pd.DataFrame([input_dict])
    input_df = input_df.reindex(columns=model_features, fill_value=0)

    prediction = rf_model.predict(input_df)[0]
    probability = rf_model.predict_proba(input_df)[0][1]

    return prediction, probability

In [ ]:
model_features = X.columns

In [ ]:
sample_input = {
    "age": 45,
    "bmi": 27,
    "vitamin_b12": 200,
    "sleep_hours": 5,
    "exercise_level": 1
}

prediction, probability = predict_numbness(sample_input)

print(prediction, probability)

In [ ]:
import joblib

joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(model_features, "model_features.pkl")

print("Saved files in current environment")

## Results
The model achieved approximately 84.5% accuracy. Vitamin B12 was the most important factor in predicting numbness.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
accuracy

In [ ]:
pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

In [ ]:
def predict_numbness(b12, iron, bmi, age):
    input_data = pd.DataFrame([{
        "vitamin_b12_percent_rda": b12,
        "iron_percent_rda": iron,
        "bmi": bmi,
        "age": age
    }])

    prediction = model.predict(input_data)

    if prediction[0] == 1:
        return "High likelihood of numbness"
    else:
        return "Low likelihood of numbness"

# Example
predict_numbness(30, 40, 28, 35)

## Future Work
This approach can be extended to cybersecurity by using system activity data to predict suspicious behavior or potential security threats.